# Pipeline de Modelagem — Clusterização K-Means por UF

### Objetivo:
Segmentar as 27 Unidades da Federação em **perfis comerciais acionáveis** para a empresa de
aquecimento de água (portfólio: coletor solar + bomba de calor + reservatório térmico),
indicando onde priorizar cada linha de produto e onde está a maior oportunidade de
conversão do chuveiro elétrico.

### Técnica:
K-Means (aprendizado não supervisionado) sobre as features do `painel_uf_integrado.csv`,
com padronização (StandardScaler) e escolha de k pelo método do cotovelo + silhueta.

In [ ]:
# ------------------------------------------------------------
# 1. MONTAGEM DO GOOGLE DRIVE
# ------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ------------------------------------------------------------
# 2. IMPORTAÇÃO DAS BIBLIOTECAS
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# ------------------------------------------------------------
# 3. CARGA DA TABELA-MESTRE POR UF
# ------------------------------------------------------------
PASTA = "/content/drive/MyDrive/dados_projeto_ciencia_dados"
df = pd.read_csv(f"{PASTA}/painel_uf_integrado.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
# ------------------------------------------------------------
# 4. SELEÇÃO DE FEATURES
# Quatro eixos de decisão comercial, um por feature, evitando
# redundância — em especial COP e temperatura, que são colineares
# (COP é função direta da temperatura): usamos apenas o COP.
# ------------------------------------------------------------
FEATURES = [
    "aptidao_solar",                 # aptidão do coletor solar (kWh/m²/dia útil)
    "cop_medio_anual",               # eficiência da bomba de calor (clima)
    "consumo_resid_por_domicilio",   # tamanho do prêmio (carga do chuveiro a converter)
    "rendimento_medio_pc",           # capacidade de compra
]
X = df[FEATURES].copy()
print(X.describe().round(2))

In [ ]:
# ------------------------------------------------------------
# 5. PADRONIZAÇÃO (StandardScaler)
# Features em escalas diferentes (R$, kWh, COP) — padronizar é
# obrigatório para o K-Means, que se baseia em distância euclidiana.
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Médias após padronização:", np.round(X_scaled.mean(axis=0), 3))
print("Desvios após padronização:", np.round(X_scaled.std(axis=0), 3))

In [ ]:
# ------------------------------------------------------------
# 6. ESCOLHA DE k — método do cotovelo (inércia) + silhueta
# ------------------------------------------------------------
ks = range(2, 8)
inercias, silhuetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(list(ks), inercias, "o-"); ax1.set_title("Método do Cotovelo")
ax1.set_xlabel("k"); ax1.set_ylabel("Inércia")
ax2.plot(list(ks), silhuetas, "o-", color="darkorange"); ax2.set_title("Silhueta")
ax2.set_xlabel("k"); ax2.set_ylabel("Coeficiente de silhueta")
plt.tight_layout(); plt.show()

for k, s in zip(ks, silhuetas):
    print(f"  k={k}: silhueta={s:.3f}")

In [ ]:
# ------------------------------------------------------------
# 7. AJUSTE DO MODELO — k = 4
# Quatro segmentos: melhor silhueta na faixa de k baixo e a
# leitura de negócio mais clara (um perfil por estratégia de venda).
# ------------------------------------------------------------
K = 4
modelo = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X_scaled)
df["cluster"] = modelo.labels_
print("Silhueta final:", round(silhouette_score(X_scaled, modelo.labels_), 3))
print(df["cluster"].value_counts().sort_index())

In [ ]:
# ------------------------------------------------------------
# 8. ROTULAGEM COMERCIAL DOS CLUSTERS (dinâmica, pelo perfil)
# Os números de cluster do K-Means são arbitrários; nomeamos
# cada um pelo seu perfil para garantir leitura estável.
# ------------------------------------------------------------
perfil = df.groupby("cluster")[FEATURES].mean()

solar_c   = perfil["aptidao_solar"].idxmax()                       # melhor sol
resto     = [c for c in perfil.index if c != solar_c]
premium_c = perfil.loc[resto, "rendimento_medio_pc"].idxmax()      # maior renda
resto2    = [c for c in resto if c != premium_c]
baixa_c   = perfil.loc[resto2, "rendimento_medio_pc"].idxmin()     # menor renda
front_c   = [c for c in resto2 if c != baixa_c][0]

NOMES = {solar_c: "Vitrine solar", premium_c: "Prêmio premium",
         baixa_c: "Baixa prioridade", front_c: "Fronteira morna"}
df["segmento"] = df["cluster"].map(NOMES)
print(NOMES)

In [ ]:
# ------------------------------------------------------------
# 9. PERFIL MÉDIO DE CADA SEGMENTO
# ------------------------------------------------------------
resumo = (
    df.groupby("segmento")
      .agg(n_ufs=("uf","count"),
           aptidao_solar=("aptidao_solar","mean"),
           cop_medio=("cop_medio_anual","mean"),
           oportunidade=("oportunidade_conversao","mean"),
           renda=("rendimento_medio_pc","mean"),
           consumo_dom=("consumo_resid_por_domicilio","mean"))
      .round(2).sort_values("oportunidade", ascending=False)
)
print(resumo.to_string())

In [ ]:
# ------------------------------------------------------------
# 10. MAPA DE PRIORIZAÇÃO — aptidão solar x COP, por segmento
# ------------------------------------------------------------
cores_seg = {"Prêmio premium":"tab:purple","Vitrine solar":"tab:orange",
             "Fronteira morna":"tab:green","Baixa prioridade":"tab:gray"}
fig, ax = plt.subplots(figsize=(10,7))
for seg, g in df.groupby("segmento"):
    ax.scatter(g.aptidao_solar, g.cop_medio_anual,
               s=g.oportunidade_conversao*6, color=cores_seg[seg], alpha=0.7, label=seg)
    for _, r in g.iterrows():
        ax.annotate(r.uf, (r.aptidao_solar, r.cop_medio_anual), fontsize=8)
ax.set_xlabel("Aptidão solar — calor útil (kWh/m²/dia)")
ax.set_ylabel("COP médio anual — bomba de calor")
ax.set_title("Segmentos Comerciais por UF\n(tamanho = oportunidade de conversão do chuveiro elétrico)")
ax.legend(title="Segmento"); plt.tight_layout(); plt.show()

In [ ]:
# ------------------------------------------------------------
# 11. ASSINATURA DOS SEGMENTOS (perfil padronizado)
# Quão acima/abaixo da média nacional cada segmento está em cada feature.
# ------------------------------------------------------------
z = (df.groupby("segmento")[FEATURES].mean() - df[FEATURES].mean()) / df[FEATURES].std()
fig, ax = plt.subplots(figsize=(9,4))
im = ax.imshow(z.values, cmap="coolwarm", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(FEATURES))); ax.set_xticklabels(FEATURES, rotation=20, ha="right")
ax.set_yticks(range(len(z))); ax.set_yticklabels(z.index)
for i in range(z.shape[0]):
    for j in range(z.shape[1]):
        ax.text(j, i, f"{z.values[i,j]:+.1f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Assinatura dos Segmentos (desvios da média nacional)")
plt.tight_layout(); plt.show()

In [ ]:
# ------------------------------------------------------------
# 12. UFs POR SEGMENTO
# ------------------------------------------------------------
for seg in ["Prêmio premium","Vitrine solar","Fronteira morna","Baixa prioridade"]:
    ufs = sorted(df[df.segmento==seg].uf.tolist())
    print(f"{seg:18s} ({len(ufs)}): {', '.join(ufs)}")

In [ ]:
# ------------------------------------------------------------
# 13. EXPORTAÇÃO — base para o dashboard
# ------------------------------------------------------------
saida = df[["cod_uf","uf","uf_nome","regiao","cluster","segmento",
            "aptidao_solar","cop_medio_anual","oportunidade_conversao",
            "rendimento_medio_pc","consumo_resid_por_domicilio",
            "trend_aquecedor_solar","trend_bomba_de_calor"]]
saida.to_csv(f"{PASTA}/painel_uf_clusters.csv", index=False, encoding="utf-8-sig")
print("painel_uf_clusters.csv exportado.")

## 14. Recomendações de negócio por segmento

| Segmento | UFs (perfil) | Recomendação comercial |
|---|---|---|
| **Prêmio premium** | Sul/Sudeste + capitais (frio + renda alta + maior oportunidade) | **Prioridade máxima.** Clima mais frio e renda alta favorecem bomba de calor e solar com apoio elétrico. Maior valor de conversão do chuveiro elétrico — foco de vendas e de maior ticket. |
| **Vitrine solar** | Nordeste (irradiação máxima, renda média) | **Empurrar coletor solar.** Melhor desempenho solar do país, mas o interesse de busca ainda é baixo → **marketing educativo** para destravar demanda (potencial técnico ocioso). |
| **Fronteira morna** | Norte/Centro-Oeste (potencial intermediário) | Testar solar onde houver renda; mercado de segundo estágio. |
| **Baixa prioridade** | Norte/NE de menor renda (clima quente, pouca necessidade) | Mercado difícil no curto prazo: baixa renda e baixa necessidade de aquecimento. Manter monitoramento. |

> Observação: a **oportunidade de conversão** e a **renda** são fortemente correlacionadas (+0,84) — os melhores alvos de conversão também são os que têm poder de compra, o que reforça a priorização do segmento Prêmio premium.